# 04 — Gold dim_customer SCD Type 2

## Overview
This lab builds and incrementally maintains `gold.dim_customer` as a **Slowly Changing Dimension Type 2 (SCD2)**. Each time a tracked attribute changes, the old row is expired (given a `valid_to` date and `is_current=false`) and a new row is inserted with today's `valid_from` — preserving the full history of every customer's attribute changes.

**What you will learn**
- The SCD2 pattern: surrogate keys, `valid_from`/`valid_to` date ranges, and `is_current` flag
- How to perform a **two-pass write** using Delta MERGE to atomically expire old rows and insert new versions
- How to detect changed rows by joining Silver against the currently-active Gold rows
- How to handle customer deletions: expired rows with `is_current=false` without losing history
- How to assert the SCD2 cardinal invariant: no customer may have more than one `is_current=true` row

**Prerequisites:** Scenario 03 has been run so `silver.customers` is populated.  
**Modes:** Set `p_mode = "init"` for the very first full load; `"incremental"` for all subsequent runs.

### Cell 1 — Run-mode parameter
`"init"` performs a **one-time full historical load** from Silver into Gold; `"incremental"` detects and applies only new changes. Making mode explicit prevents accidental re-initialisation in production — running `init` on an existing Gold table would overwrite all SCD2 history accumulated since the last full load.

In [ ]:
p_mode = "incremental"   # 'init' | 'incremental'

### Cell 2 — Configuration & tracked columns
`TRACKED_COLS` is the key design decision: it declares which Silver columns — when their value changes — should trigger a new SCD2 version. Only meaningful business attributes (`name`, `region`, `email_hash`) are tracked; internal audit columns like `_silver_ts` are intentionally excluded so routine pipeline re-runs do not spuriously generate new Gold rows. `SK_OFFSET` namespaces surrogate keys away from other dimension tables to prevent cross-table collisions in conformed star schemas.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date

LH          = "lh_advanced_scenarios"
SILVER      = f"{LH}.silver.customers"
GOLD        = f"{LH}.gold.dim_customer"
SK_OFFSET   = 1_000_000   # surrogate key offset
TODAY       = date.today().isoformat()
YESTERDAY   = (date.today().__class__.fromordinal(date.today().toordinal() - 1)).isoformat()

# Columns that trigger a new SCD2 version when changed
TRACKED_COLS = ["name", "region", "email_hash"]

### Cell 3 — Read Silver (all rows including deleted)
Loads the full Silver customer table — including soft-deleted rows. Deleted customers are needed because a deletion event must also be reflected in Gold: the current Gold row must be expired, and a final `is_current=false` version is inserted. Excluding deleted Silver rows from this read would leave stale `is_current=true` Gold rows that misrepresent the customer base.

In [ ]:
# ── Read Silver (active + deleted rows) ──────────────────────────────────────
df_silver = spark.read.format("delta").table(SILVER)

### Cell 4 — Init path: full first load
Creates the Gold dimension table from all current Silver rows. Every row starts as version 1: `valid_from=today`, `valid_to=NULL` (open-ended — no expiry date yet), `is_current=true`. `monotonically_increasing_id()` generates unique surrogate keys for this batch; the `SK_OFFSET` separates them from other dimensions. The notebook exits immediately after so the incremental path below does not also execute.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# INIT MODE: first full load
# ════════════════════════════════════════════════════════════════════════════
if p_mode == "init" or not spark.catalog.tableExists(GOLD):
    df_init = (
        df_silver
        .withColumn("customer_sk", F.monotonically_increasing_id() + SK_OFFSET)
        .withColumn("valid_from",  F.lit(TODAY).cast("date"))
        .withColumn("valid_to",    F.lit(None).cast("date"))
        .withColumn("is_current",  F.lit(True))
        .withColumn("_gold_ts",    F.current_timestamp())
        .select(
            "customer_sk", "customer_id", "name", "email_hash", "region",
            "valid_from", "valid_to", "is_current", "is_deleted", "_gold_ts"
        )
    )
    df_init.write.format("delta").mode("overwrite") \
           .option("overwriteSchema", "true") \
           .saveAsTable(GOLD)
    print(f"Init load complete. Rows: {df_init.count():,}")
    dbutils.notebook.exit("Init complete.")

### Cell 5 — Detect changed & new rows
Performs a **left join** of Silver against the currently-active Gold rows (`is_current=true`). The filter keeps rows where: (a) no matching Gold row exists yet (`t.customer_id IS NULL` — new customers), (b) any tracked column value differs between Silver and Gold, or (c) the `is_deleted` flag has changed. The dynamically-built `change_condition` string evaluates each `TRACKED_COLS` column, including NULL-safe comparison to catch `NULL → value` transitions.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# INCREMENTAL MODE
# ════════════════════════════════════════════════════════════════════════════

# Read current Gold rows
df_gold_current = (
    spark.read.format("delta").table(GOLD)
         .filter(F.col("is_current") == True)
)

# Detect changed or new rows
change_condition = " OR ".join(
    [f"s.{c} <> t.{c} OR (s.{c} IS NULL AND t.{c} IS NOT NULL)" for c in TRACKED_COLS]
)

df_changed = (
    df_silver.alias("s")
             .join(df_gold_current.alias("t"), "customer_id", "left")
             .filter(f"t.customer_id IS NULL OR ({change_condition}) OR s.is_deleted != t.is_deleted")
             .select("s.*")
)

print(f"Changed/new rows: {df_changed.count():,}")

### Cell 6 — Pass 1: expire current rows that changed
For every customer in the changed set, the **currently-active Gold row** is closed: `valid_to` is set to yesterday (one day before the new version's `valid_from`), and `is_current` is set to `false`. This creates a contiguous, non-overlapping date range. The join condition `t.is_current = true` ensures only the live row is targeted — historical versions are left untouched.

In [ ]:
# ── Pass 1: expire existing current rows that changed ────────────────────────
gold_dt = DeltaTable.forName(spark, GOLD)
(
    gold_dt.alias("t")
           .merge(
               df_changed.alias("s"),
               "t.customer_id = s.customer_id AND t.is_current = true"
           )
           .whenMatchedUpdate(set={
               "valid_to":   f"date_sub(current_date(), 1)",
               "is_current": "false",
               "_gold_ts":   "current_timestamp()",
           })
           .execute()
)
print("Pass 1 (expire) complete.")

### Cell 7 — Pass 2: insert new versions
Appends a **new Gold row** for each changed customer reflecting their current Silver state. The surrogate key is guaranteed unique by reading the current max SK and incrementing from it. Deleted customers receive `is_current=false` immediately — their new version is a tombstone row that exists purely for auditing. Non-deleted changed customers get `is_current=true`.

In [ ]:
# ── Pass 2: insert new versions for changed rows ──────────────────────────────
max_sk = spark.sql(f"SELECT MAX(customer_sk) AS m FROM {GOLD}").collect()[0][0] or SK_OFFSET

df_new_versions = (
    df_changed
    .withColumn("customer_sk", F.monotonically_increasing_id() + max_sk + 1)
    .withColumn("valid_from",  F.lit(TODAY).cast("date"))
    .withColumn("valid_to",    F.lit(None).cast("date"))
    .withColumn("is_current",  F.when(F.col("is_deleted") == True, F.lit(False)).otherwise(F.lit(True)))
    .withColumn("_gold_ts",    F.current_timestamp())
    .select(
        "customer_sk", "customer_id", "name", "email_hash", "region",
        "valid_from", "valid_to", "is_current", "is_deleted", "_gold_ts"
    )
)

df_new_versions.write.format("delta").mode("append").saveAsTable(GOLD)
print(f"Pass 2 (insert new versions) complete. Rows inserted: {df_new_versions.count():,}")

### Cell 8 — SCD2 invariant validation
Asserts that no `customer_id` has more than one row with `is_current=true`. This is the **cardinal SCD2 invariant**: any violation means a Power BI report filtering `WHERE is_current = 1` would return duplicate customer rows, corrupting every metric that joins to this dimension. The assertion fails the notebook — and the Data Factory pipeline — if broken.

In [ ]:
# ── Validation ───────────────────────────────────────────────────────────────
multi_current = spark.sql(
    f"SELECT customer_id, SUM(CAST(is_current AS INT)) c FROM {GOLD} "
    f"GROUP BY customer_id HAVING c > 1"
).count()
assert multi_current == 0, f"{multi_current} customers have >1 current row"

total = spark.sql(f"SELECT COUNT(*) c FROM {GOLD}").collect()[0][0]
print(f"dim_customer total rows: {total:,} | Multi-current violations: {multi_current}")